# Reliability — Publication Stats & Figures (deep ensemble vs single-model)

The bankable result: on this accuracy-saturated task, **deep-ensemble uncertainty gives significantly
better-calibrated, more reliable failure detection than single-model confidence.** Established across
42 held-out cases — AUROC 0.795 vs 0.743, ECE 0.075 vs 0.156, risk-coverage 0.062 vs 0.098, all p<1e-4.

This notebook produces the **publication-ready outputs**, all saved to Drive:
- per-case **paired significance** (Wilcoxon + effect size) for ensemble vs entropy on AUROC / ECE / risk-coverage
- **risk–coverage curves** (the headline selective-prediction figure)
- a **reliability diagram** (calibration) for ensemble vs single model
- consolidated tables + summary text

Part A re-derives significance from the existing per-case CSV (instant). Part B regenerates the curves
from checkpoints (inference only, no training). Run Part A alone if you just need the stats.

## A1. Config / mount

In [ ]:
from google.colab import drive; drive.mount('/content/drive')
from pathlib import Path
import numpy as np, torch
DRIVE=Path("/content/drive/MyDrive")

class C:
    cache_root = DRIVE/"Datasets"/"PancreasCache"      # same cache used for training
    ckpt_dir   = DRIVE/"Outputs"/"Pancreas-Compare"/"Checkpoints"
    out_dir    = DRIVE/"Outputs"/"Pancreas-Reliability"
    seeds      = (0,1,2,3,4)
    test_seed  = 12345           # MUST match the training run (same fixed 38/42-case test set)
    test_fraction = 0.15
    target_shape = (112,112,112)
    base_channels = 16
    n_dist = 3                   # Fresnel propagation distances (match training)
    assess_arch = "fresnel"      # whose predictions we assess: "fresnel" or "baseline"
    tta_K = 6
    device = "cuda" if torch.cuda.is_available() else "cpu"
cfg=C()
# optional: copy cache to local disk to dodge Drive read flakiness
import shutil
loc=Path("/content/PancreasCache")
if not loc.exists() and cfg.cache_root.exists():
    shutil.copytree(cfg.cache_root, loc); cfg.cache_root=loc; print("cache -> local disk")
elif loc.exists(): cfg.cache_root=loc
OUT=cfg.out_dir; (OUT/"Tables").mkdir(parents=True,exist_ok=True); (OUT/"Figures").mkdir(parents=True,exist_ok=True)
print("device:",cfg.device,"| ckpts:",cfg.ckpt_dir)


## A2. Per-case significance from existing reliability CSV (instant, no inference)

In [ ]:
import pandas as pd, numpy as np, json
from scipy import stats
PC = OUT/"Tables"/"reliability_per_case.csv"
assert PC.exists(), f"missing {PC} — run Reliability_Eval first"
d=pd.read_csv(PC)
def paired(metric, ref="entropy", arm="ensemble", lower_better=False):
    piv=d.pivot_table(index="case",columns="method",values=metric)
    a,b=piv[arm].dropna(),piv[ref].dropna(); idx=a.index.intersection(b.index); a,b=a[idx],b[idx]
    diff=a-b
    try: p=float(stats.wilcoxon(a,b).pvalue)
    except Exception: p=float("nan")
    # rank-biserial effect size
    nz=diff[diff!=0]; 
    rbc=float(np.sign(nz).mean()) if len(nz) else 0.0
    wins=int((diff<0).sum() if lower_better else (diff>0).sum())
    return dict(metric=metric,arm=arm,ref=ref,arm_mean=round(float(a.mean()),4),ref_mean=round(float(b.mean()),4),
                delta=round(float(diff.mean()),4),p_value=p,rank_biserial=round(rbc,3),
                wins=wins,n=len(diff),lower_better=lower_better)
rows=[paired("auroc"),paired("ece",lower_better=True),paired("rcauc",lower_better=True)]
# also vs tta if present
if "tta" in d.method.unique():
    rows+=[paired("auroc",ref="tta"),paired("ece",ref="tta",lower_better=True)]
sig=pd.DataFrame(rows)
# Holm across the 3 primary ensemble-vs-entropy tests
prim=sig[sig.ref=="entropy"].copy().reset_index(drop=True)
order=prim.p_value.values.argsort(); m=len(order); holm=np.empty(m)
for rank,i in enumerate(order): holm[i]=min(1.0,prim.p_value.values[i]*(m-rank))
prim["holm_p"]=holm.round(6); prim["sig"]=prim.holm_p<0.05
(OUT/"Tables").mkdir(parents=True,exist_ok=True)
sig.to_csv(OUT/"Tables"/"reliability_significance.csv",index=False)
prim.to_csv(OUT/"Tables"/"reliability_significance_primary.csv",index=False)
print("Ensemble vs single-model entropy (paired Wilcoxon, n=42), Holm-corrected:")
for _,r in prim.iterrows():
    arrow="↑" if not r.lower_better else "↓"
    print(f"  {r.metric:<6} ens={r.arm_mean} vs ent={r.ref_mean}  Δ={r.delta:+.4f} {arrow}  "
          f"p={r.p_value:.2e}  Holm={r.holm_p:.2e}  wins {r.wins}/{r.n}  {'SIG' if r.sig else 'ns'}")


## B1. (figures) Data + models for curve regeneration

In [ ]:
cache_files=sorted(p for p in cfg.cache_root.glob("*.npz"))
allidx=np.arange(len(cache_files)); np.random.default_rng(cfg.test_seed).shuffle(allidx)
ntest=int(len(cache_files)*cfg.test_fraction)
test_files=[cache_files[i] for i in allidx[:ntest]]
def detect(files,sample=60):
    vals=set()
    for p in files[:sample]: vals.update(int(v) for v in np.unique(np.load(p)["lab"]) if v>0)
    fg=sorted(vals); return {v:i+1 for i,v in enumerate(fg)}, len(fg)
LMAP,NFG=detect(cache_files); NCLS=NFG+1
CLASS_NAMES={1:"pancreas",2:"mass"} if NFG==2 else {c:f"class{c}" for c in range(1,NCLS)}
def load_case(p):
    d=np.load(p); img=d["img"].astype(np.float32); raw=d["lab"].astype(np.int64)
    img=(img-img.mean())/(img.std()+1e-6)
    lab=np.zeros_like(raw); 
    for v,k in LMAP.items(): lab[raw==v]=k
    return img.astype(np.float32), lab
print(f"{NCLS}-way {CLASS_NAMES} | fixed test cases: {len(test_files)}")


In [ ]:
import math, torch.nn as nn, torch.nn.functional as F

class MultiBandSpectralEdgeAttention3D(nn.Module):
    def __init__(s,ch,nb=4):
        super().__init__(); s.mu=nn.Parameter(torch.linspace(0.45,0.9,nb)); s.log_sigma=nn.Parameter(torch.log(torch.full((nb,),0.08)))
        s.fuse=nn.Conv3d(ch*nb,ch,1); s.gate=nn.Sequential(nn.Conv3d(ch,ch,1),nn.Sigmoid()); s.n=nb
    def _rho(s,D,H,W,dev):
        fz=torch.fft.fftfreq(D,device=dev);fy=torch.fft.fftfreq(H,device=dev);fx=torch.fft.rfftfreq(W,device=dev)
        Z,Y,X=torch.meshgrid(fz,fy,fx,indexing="ij"); r=torch.sqrt(Z**2+Y**2+X**2); return r/(r.max()+1e-6)
    def forward(s,x):
        D,H,W=x.shape[-3:];Xf=torch.fft.rfftn(x.float(),dim=(-3,-2,-1));rho=s._rho(D,H,W,x.device)
        sg=torch.exp(s.log_sigma);mu=torch.clamp(s.mu,0,1)
        b=[torch.fft.irfftn(Xf*torch.exp(-((rho-mu[i])**2)/(2*sg[i]**2+1e-6)).to(Xf.dtype),s=(D,H,W),dim=(-3,-2,-1)).to(x.dtype) for i in range(s.n)]
        return x*(1+s.gate(s.fuse(torch.cat(b,1))))

class FresnelPropagation3D(nn.Module):
    def __init__(s,ch,nd=3):
        super().__init__(); s.z=nn.Parameter(torch.linspace(0.05,0.30,nd)); s.fuse=nn.Conv3d(ch*nd,ch,1); s.n=nd; s.capture=False; s.last=None
    def _k2(s,D,H,W,dev):
        fz=torch.fft.fftfreq(D,device=dev);fy=torch.fft.fftfreq(H,device=dev);fx=torch.fft.rfftfreq(W,device=dev)
        Z,Y,X=torch.meshgrid(fz,fy,fx,indexing="ij"); return Z**2+Y**2+X**2
    def forward(s,x):
        D,H,W=x.shape[-3:];Xf=torch.fft.rfftn(x.float(),dim=(-3,-2,-1));k2=s._k2(D,H,W,x.device)
        outs=[torch.fft.irfftn(Xf*torch.exp(1j*(2*math.pi)*s.z[i]*k2).to(Xf.dtype),s=(D,H,W),dim=(-3,-2,-1)).to(x.dtype) for i in range(s.n)]
        if s.capture: s.last=torch.stack(outs,0).detach()   # (n_dist, B, C, D,H,W)
        return x+s.fuse(torch.cat(outs,1))

class ResBlock3D(nn.Module):
    def __init__(s,ci,co):
        super().__init__();s.c1=nn.Conv3d(ci,co,3,padding=1);s.n1=nn.InstanceNorm3d(co);s.c2=nn.Conv3d(co,co,3,padding=1);s.n2=nn.InstanceNorm3d(co);s.skip=nn.Conv3d(ci,co,1) if ci!=co else nn.Identity()
    def forward(s,x):h=torch.relu(s.n1(s.c1(x)));h=s.n2(s.c2(h));return torch.relu(h+s.skip(x))
def make_module(a,ch,cfg):
    if a=="sea": return MultiBandSpectralEdgeAttention3D(ch,4)
    if a=="fresnel": return FresnelPropagation3D(ch,cfg.n_dist)
    return nn.Identity()
class CompareUNet3D(nn.Module):
    def __init__(s,cfg,n_classes,arch="baseline",in_ch=1):
        super().__init__();base=cfg.base_channels
        s.e1=ResBlock3D(in_ch,base);s.e2=ResBlock3D(base,2*base);s.bott=ResBlock3D(2*base,4*base)
        s.up2=nn.ConvTranspose3d(4*base,2*base,2,2);s.d2=ResBlock3D(4*base,2*base)
        s.up1=nn.ConvTranspose3d(2*base,base,2,2);s.d1=ResBlock3D(2*base,base)
        s.m2=make_module(arch,2*base,cfg);s.m1=make_module(arch,base,cfg)
        s.out=nn.Conv3d(base,n_classes,1);s.pool=nn.MaxPool3d(2)
    def forward(s,x):
        e1=s.e1(x);e2=s.e2(s.pool(e1));b=s.bott(s.pool(e2))
        d2=s.m2(s.d2(torch.cat([s.up2(b),e2],1)));d1=s.m1(s.d1(torch.cat([s.up1(d2),e1],1)))
        return s.out(d1)
def build(arch):
    return CompareUNet3D(cfg,NCLS,arch).to(cfg.device)
def load_seed(arch,seed):
    net=build(arch); p=cfg.ckpt_dir/f"{arch}_seed{seed}.pt"
    net.load_state_dict(torch.load(p,map_location=cfg.device)); net.eval(); return net
print("model defs ready")


In [ ]:
from scipy.ndimage import binary_dilation

def softmax_np(logits): 
    e=np.exp(logits-logits.max(0,keepdims=True)); return e/e.sum(0,keepdims=True)

@torch.no_grad()
def logits_of(net,img):
    x=torch.from_numpy(img)[None,None].to(cfg.device)
    return net(x)[0].cpu().numpy()            # (C,D,H,W)

@torch.no_grad()
def fresnel_coherence(net,img):
    # one forward pass; variance across n_dist propagated views at the final Fresnel layer
    for m in net.modules():
        if isinstance(m,FresnelPropagation3D): m.capture=True; m.last=None
    x=torch.from_numpy(img)[None,None].to(cfg.device); lo=net(x)[0].cpu().numpy()
    caps=[m.last for m in net.modules() if isinstance(m,FresnelPropagation3D) and m.last is not None]
    m_last=caps[-1]                            # (n_dist,B,C,d,h,w) at full-ish res (base channels)
    v=m_last.float().var(0).mean(1)[0]         # var across distances, mean over channels -> (d,h,w)
    u=torch.from_numpy(v.cpu().numpy()) if hasattr(v,'cpu') else v
    u=v.cpu().numpy()
    # upsample to full res
    u=torch.from_numpy(u)[None,None].float()
    u=F.interpolate(u,size=img.shape,mode="trilinear",align_corners=False)[0,0].numpy()
    for m in net.modules():
        if isinstance(m,FresnelPropagation3D): m.capture=False; m.last=None
    return lo,u

@torch.no_grad()
def tta_uncertainty(net,img,K=6):
    sm=[]; 
    base=softmax_np(logits_of(net,img)); sm.append(base)
    for k in range(K-1):
        aug=img+np.random.normal(0,0.03,img.shape).astype(np.float32)
        flips=[ax for ax in (0,1,2) if (k>>ax)&1]
        a=aug.copy()
        for ax in flips: a=np.flip(a,ax)
        s=softmax_np(logits_of(net,np.ascontiguousarray(a)))
        for ax in flips: s=np.flip(s,ax+1)
        sm.append(np.ascontiguousarray(s))
    sm=np.stack(sm,0)                          # (K,C,D,H,W)
    pbar=sm.mean(0); ent=-(pbar*np.log(pbar+1e-9)).sum(0)   # predictive entropy
    return pbar, ent

def entropy_u(prob): return -(prob*np.log(prob+1e-9)).sum(0)

# ---- metrics on an evaluation region R (band around structure) ----
def eval_region(pred,gt):
    fg=(pred>0)|(gt>0)
    return binary_dilation(fg,iterations=3)

def auroc(u,err):
    # Mann-Whitney AUROC: P(u_err > u_correct)
    pos=u[err==1]; neg=u[err==0]
    if len(pos)==0 or len(neg)==0: return np.nan
    # rank-based
    allv=np.concatenate([pos,neg]); order=allv.argsort(kind="mergesort"); ranks=np.empty(len(allv)); ranks[order]=np.arange(1,len(allv)+1)
    # average ties
    # (approx: ignore ties for speed on large arrays via argsort ranks)
    rp=ranks[:len(pos)].sum(); 
    return (rp-len(pos)*(len(pos)+1)/2)/(len(pos)*len(neg))

def ece(conf,correct,bins=10):
    e=0.0; n=len(conf)
    for i in range(bins):
        lo,hi=i/bins,(i+1)/bins; m=(conf>lo)&(conf<=hi)
        if m.sum()==0: continue
        e+=abs(correct[m].mean()-conf[m].mean())*m.sum()/n
    return e

def risk_coverage_auc(conf,correct,steps=20):
    order=np.argsort(-conf)  # most confident first
    c=correct[order]; aucs=[]
    for q in np.linspace(0.1,1.0,steps):
        k=max(1,int(len(c)*q)); aucs.append(1-c[:k].mean())   # risk = error rate at coverage q
    return float(np.mean(aucs))
print("estimators + metrics ready")


## B2. Risk–coverage curves + reliability diagram (inference; saved to Drive)

In [ ]:
import matplotlib.pyplot as plt
fres=[load_seed("fresnel",s) for s in cfg.seeds]; m0=fres[0]

# accumulate (confidence, correct) over all test voxels in region R, for single-model vs ensemble
def collect():
    s_conf=[]; s_corr=[]; e_conf=[]; e_corr=[]
    for p in test_files:
        img,gt=load_case(p)
        p0=softmax_np(logits_of(m0,img)); pred0=p0.argmax(0); R=eval_region(pred0,gt).astype(bool)
        s_conf.append(p0.max(0)[R]); s_corr.append((pred0[R]==gt[R]).astype(np.float32))
        pbar=np.mean([softmax_np(logits_of(n,img)) for n in fres],0); prede=pbar.argmax(0)
        e_conf.append(pbar.max(0)[R]); e_corr.append((prede[R]==gt[R]).astype(np.float32))
    return (np.concatenate(s_conf),np.concatenate(s_corr),np.concatenate(e_conf),np.concatenate(e_corr))
sc,sco,ec,eco=collect()

def risk_cov(conf,corr,steps=25):
    order=np.argsort(-conf); c=corr[order]; cov=np.linspace(0.04,1,steps); risk=[1-c[:max(1,int(len(c)*q))].mean() for q in cov]; return cov,np.array(risk)
def reliability(conf,corr,bins=10):
    xs,ys=[],[]
    for i in range(bins):
        lo,hi=i/bins,(i+1)/bins; m=(conf>lo)&(conf<=hi)
        if m.sum()>0: xs.append(conf[m].mean()); ys.append(corr[m].mean())
    return np.array(xs),np.array(ys)

# Fig 1: risk-coverage
fig,ax=plt.subplots(figsize=(6,4))
cov,r_s=risk_cov(sc,sco); _,r_e=risk_cov(ec,eco)
ax.plot(cov,r_s,"-o",ms=3,label="single model (entropy)",color="#999999")
ax.plot(cov,r_e,"-o",ms=3,label="deep ensemble",color="#4477aa")
ax.set_xlabel("coverage (fraction retained, most-confident first)"); ax.set_ylabel("risk (error rate)")
ax.set_title("Risk–coverage: ensemble defers better"); ax.legend(); ax.grid(alpha=.3)
plt.tight_layout(); fig.savefig(OUT/"Figures"/"risk_coverage_curve.png",dpi=160,bbox_inches="tight"); plt.show()

# Fig 2: reliability diagram
fig,ax=plt.subplots(figsize=(5,5))
ax.plot([0,1],[0,1],"k--",lw=.8,label="perfect")
xs,ys=reliability(sc,sco); ax.plot(xs,ys,"-o",ms=4,label="single model",color="#999999")
xs,ys=reliability(ec,eco); ax.plot(xs,ys,"-o",ms=4,label="deep ensemble",color="#4477aa")
ax.set_xlabel("confidence"); ax.set_ylabel("accuracy"); ax.set_title("Reliability diagram"); ax.legend(); ax.grid(alpha=.3)
plt.tight_layout(); fig.savefig(OUT/"Figures"/"reliability_diagram.png",dpi=160,bbox_inches="tight"); plt.show()

np.savez(OUT/"Tables"/"curve_data.npz",cov=cov,risk_single=r_s,risk_ensemble=r_e)
print("saved risk_coverage_curve.png, reliability_diagram.png, curve_data.npz ->",OUT)


## C. Consolidated summary -> Drive

In [ ]:
lines=["PANCREAS — RELIABILITY: deep ensemble vs single-model (publication stats)","="*66,
       f"test cases: {len(test_files) if 'test_files' in dir() else 42} | seeds: {cfg.seeds}","",
       "Per-case paired Wilcoxon (Holm-corrected), ensemble vs single-model entropy:"]
for _,r in prim.iterrows():
    arrow="higher better" if not r.lower_better else "lower better"
    lines.append(f"  {r.metric:<6} ({arrow}): ensemble={r.arm_mean}  entropy={r.ref_mean}  Δ={r.delta:+.4f}  "
                 f"Holm p={r.holm_p:.2e}  wins {r.wins}/{r.n}  {'SIGNIFICANT' if r.sig else 'ns'}")
lines+=["","Interpretation: ensemble disagreement is a significantly better-calibrated and more reliable",
        "failure detector than single-model confidence on this accuracy-saturated task. Physics-motivated",
        "single-pass proxies (fresnel_coh 0.474, fresnel_disagree 0.644) underperform entropy (0.743).",
        "UVIF: ensemble disagreement is a concrete, validated estimator for the uncertainty (U) term."]
(OUT/"reliability_publication_summary.txt").write_text("\n".join(lines)+"\n")
print("\n".join(lines)); print("\nALL SAVED TO DRIVE ->",OUT)
